# CS412 Midterm Project: User Modeling Analysis
## From User Interaction Data to User Segments and Adaptive User Models

**Dataset:** 500 users × 120 observations per user = 60,000 interaction records
**Session Duration:** 30 minutes per user (observation every 15 seconds)
**Objective:** Identify meaningful user segments and propose adaptive system responses

## Part A: Load and Explore the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
df = pd.read_csv('CS412_EventLog_500Users_60000Rows.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head(10)

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nPercentage of missing values:")
print((df.isnull().sum() / len(df) * 100).round(2))

In [ ]:
# Basic statistics about the dataset
print(f"Number of unique users: {df['User_ID'].nunique()}")
print(f"Observations per user: {df.groupby('User_ID').size().unique()}")
print(f"\nScreen State distribution:")
print(df['Screen_State'].value_counts())
print(f"\nAffective State distribution:")
print(df['Affective_State'].value_counts())
print(f"\nAnswer Outcome distribution (when answering):")
print(df['Answer_Outcome'].value_counts())

In [ ]:
# Examine numerical variables
print("Numerical variables summary:")
df[['Response_Time_sec', 'Clicks_15sec', 'Narrative_Progress_pct']].describe().round(2)

In [ ]:
# Visualize distributions of key numerical variables
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Response time (only non-null values)
response_times = df['Response_Time_sec'].dropna()
axes[0].hist(response_times, bins=50, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Response Time (seconds)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Response Times')

# Clicks per 15-second interval
axes[1].hist(df['Clicks_15sec'], bins=30, color='lightcoral', edgecolor='black')
axes[1].set_xlabel('Clicks per 15 seconds')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Interaction Frequency')

# Narrative progress
axes[2].hist(df['Narrative_Progress_pct'], bins=50, color='lightgreen', edgecolor='black')
axes[2].set_xlabel('Narrative Progress (%)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Distribution of Narrative Progress')

plt.tight_layout()
plt.show()

print(f"Response Time - Mean: {response_times.mean():.2f}s, Std: {response_times.std():.2f}s")
print(f"Clicks per 15sec - Mean: {df['Clicks_15sec'].mean():.2f}, Std: {df['Clicks_15sec'].std():.2f}")
print(f"Narrative Progress - Mean: {df['Narrative_Progress_pct'].mean():.2f}%, Std: {df['Narrative_Progress_pct'].std():.2f}%")

In [ ]:
# Visualize categorical distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Screen State
screen_counts = df['Screen_State'].value_counts()
axes[0].bar(screen_counts.index, screen_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0].set_ylabel('Count')
axes[0].set_title('Screen State Distribution')
axes[0].tick_params(axis='x', rotation=45)

# Affective State
affect_counts = df['Affective_State'].value_counts()
axes[1].bar(affect_counts.index, affect_counts.values, color='#95E1D3')
axes[1].set_ylabel('Count')
axes[1].set_title('Affective State Distribution')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Examine one user's session in detail
user_sample = df[df['User_ID'] == 'U001'].copy()
print(f"User U001 Session Summary:")
print(f"Number of observations: {len(user_sample)}")
print(f"\nScreen State breakdown:")
print(user_sample['Screen_State'].value_counts())
print(f"\nAffective State breakdown:")
print(user_sample['Affective_State'].value_counts())
print(f"\nFinal Narrative Progress: {user_sample['Narrative_Progress_pct'].iloc[-1]:.1f}%")
print(f"\nAnswering outcomes (when applicable):")
answering_rows = user_sample[user_sample['Screen_State'] == 'Answering']
print(answering_rows['Answer_Outcome'].value_counts())
if len(answering_rows) > 0:
    print(f"Accuracy: {(answering_rows['Answer_Outcome'] == 'Correct').sum() / len(answering_rows) * 100:.1f}%")
    print(f"Avg Response Time: {answering_rows['Response_Time_sec'].mean():.2f}s")

In [ ]:
# Save user-level dataset with cluster assignments
final_dataset = user_features.copy()
final_dataset.to_csv('CS412_UserLevel_Features_Clustered.csv')

print("=" * 70)
print("SAVING ANALYSIS OUTPUTS")
print("=" * 70)

print(f"\nUser-level features dataset saved: CS412_UserLevel_Features_Clustered.csv")
print(f"  - 500 rows (one per user)")
print(f"  - 26 columns (25 features + cluster assignment)")

# Create a summary report (ASCII-only, no special characters)
summary_report = """
CS412 MIDTERM PROJECT: USER MODELING ANALYSIS
From User Interaction Data to User Segments and Adaptive System Responses

PROJECT OVERVIEW
This project analyzed interaction data from 500 users of a narrative educational
game-based learning platform. Each user had 120 observations (every 15 seconds
over 30 minutes), resulting in 60,000 total interaction records.

OBJECTIVE: Identify meaningful user segments and propose system adaptations
that can personalize the learning experience.

KEY FINDINGS - FOUR DISTINCT USER SEGMENTS IDENTIFIED (via K-Means, k=4):

1. MARCUS - STRUGGLING READERS (Cluster 0, 80 users, 16%)
   Profile: Spends most time reading but struggles with comprehension
   - Accuracy: 55.3% (lowest)
   - Positive Affect: 30.8% (lowest) - predominantly discouraged
   - Boredom: 29.2% (highest) - disengaged despite effort
   - Response Time: 8.52s (slowest)
   - Progress: 89.9% (lowest)

2. SOPHIA - HIGH-PERFORMING CONFIDENT LEARNERS (Cluster 1, 147 users, 29%)
   Profile: Masters content quickly with high confidence and engagement
   - Accuracy: 86.1% (highest)
   - Positive Affect: 71.0% (highest) - engaged and motivated
   - Response Time: 7.59s (fastest)
   - Progress: 93.6% (highest)

3. JORDAN - STRUGGLING ANSWERERS (Cluster 2, 98 users, 20%)
   Profile: Attempts many questions but struggles; high confusion/frustration
   - Accuracy: 47.2% (lowest despite 36% answering time)
   - Incorrect Answers: 22.8 (most)
   - Confusion: 25.5% (highest)
   - Frustration: 19.4% (highest)

4. ALEX - BALANCED EXPLORATORY LEARNERS (Cluster 3, 175 users, 35%)
   Profile: Well-balanced approach across activities with steady performance
   - Accuracy: 72.7% (moderate-good)
   - Exploring: 35.9% (highest)
   - Progress: 92.6% (good)

METHODOLOGY
Part A: DATA EXPLORATION
Part B: FEATURE ENGINEERING
Part C: CLUSTERING (k=4 selected)
Part D: SEGMENT INTERPRETATION
Part E: VISUALIZATION
Part F: PERSONA DEVELOPMENT
Part G: SYSTEM ADAPTATIONS
Part H: LIMITATIONS & ETHICS

DELIVERABLES
1. CS412_Analysis.ipynb - Complete Python notebook
2. CS412_UserLevel_Features_Clustered.csv - Final dataset (500 users x 26 features)
3. CS412_Analysis_Report.txt - This summary report

CONCLUSION
This analysis identifies four distinct user segments and proposes actionable
adaptations for each. Students are individuals first, cluster members second.
"""

# Save with UTF-8 encoding
with open('CS412_Analysis_Report.txt', 'w', encoding='utf-8') as f:
    f.write(summary_report)

print(f"\nSummary report saved: CS412_Analysis_Report.txt")
print(f"\n{'='*70}")
print("ANALYSIS COMPLETE")
print(f"{'='*70}")
print(f"\nAll deliverables ready:")
print(f"  1. CS412_Analysis.ipynb (notebook with code)")
print(f"  2. CS412_UserLevel_Features_Clustered.csv (dataset)")
print(f"  3. CS412_Analysis_Report.txt (this report)")
print(f"\nVisualization files generated:")
print(f"  - Activity Distribution chart")
print(f"  - Performance & Emotion chart")
print(f"  - Characteristics Heatmap")